# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Dataset metadata (access via attributes, not as dictionary keys)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets (`@id`), and for each, the fields and their `@id`s.

In [ ]:
# List all record sets and fields by their @id
print("Available record sets and fields (by @id):\n")
record_set_ids = []
for record_set in dataset.metadata.record_sets:
    print(f"Record Set: {record_set['@id']}  | name: {record_set.get('name', '')}")
    record_set_ids.append(record_set['@id'])
    print("  Fields:")
    for field in record_set.get('fields', []):
        print(f"    Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")
    print()

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis.
We will use the `@id` fields from the record set above for all references.

In [ ]:
# Assign record set @id(s)—adjust these based on the dataset's reported recordSets
record_sets = record_set_ids  # Use all found record_set @ids
dataframes = {}

for record_set_id in record_sets:
    try:
        # Load records as list of dict
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame from record set {record_set_id}. Columns: {df.columns.tolist()}")
        else:
            print(f"Record set {record_set_id} returned no records.")
    except Exception as e:
        print(f"Failed to load records from {record_set_id}: {e}")

# Display first loaded DataFrame (if exists)
if dataframes:
    first_set = list(dataframes.keys())[0]
    print(f"\nFirst 5 rows from record set {first_set}:")
    display(dataframes[first_set].head())
else:
    print("No record sets loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply basic analyses: filtering, normalization, and grouping.

All field references are by `@id` as previously shown.

In [ ]:
# Choose a record set to analyze (use the first loaded one as an example if unsure)
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    print(f"Working with record set: {record_set_id}")
    # List column ids for numeric field choice
    print("Available columns:", df.columns.tolist())
    
    # Try auto-selecting a numeric field
    numeric_field = None
    for col in df.columns:
        # Try to find a float or int column
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
            # Try to coerce
            pd.to_numeric(df[col].dropna(), errors='raise')
            numeric_field = col
            break
        except:
            continue
    if numeric_field is not None:
        # Ensure column is numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        # Remove NA
        df = df.dropna(subset=[numeric_field])

        threshold = df[numeric_field].quantile(0.5)  # Use median as threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        norm_field = f"{numeric_field}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_field]].head())

        # Try to find a candidate group-by field (string/categorical)
        group_field = None
        for col in df.columns:
            if col == numeric_field:
                continue
            if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < len(df) // 2:
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if grouping is available, compare group differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field is not None and group_field in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} across {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
We loaded and explored the FAIR² dataset, identified potential numeric fields for scientific investigation, applied basic filtering and normalization, grouped by categorical attributes, and visualized distributions. The use of Croissant `@id` for all references ensures reproducible, standards-based workflows for data discovery and machine learning applications. 

Further analysis can be performed based on specific protein analysis questions or extended field selection depending on research goals.